In [1]:
from qiskit import QuantumCircuit
import json
import qiskit
from qiskit.transpiler import CouplingMap, PassManager
from typing import List
import rustworkx as rx
import numpy as np

import sys
sys.path.append('..')

from phoenix.utils.render import print_circ_info
from phoenix.primitive.grouping import group_paulis_and_coeffs

All2all = CouplingMap(rx.generators.complete_graph(35).to_directed().edge_list())

In [9]:
def gene_chain_coupling_map(size):
    return CouplingMap.from_line(size)


def gene_square_coupling_map(size):
    n = int(np.sqrt(size))
    m = int(np.ceil(size / n))
    g = rx.generators.grid_graph(n, m).subgraph(range(size)).to_directed()
    return CouplingMap(g.edge_list())

In [10]:

def is_all2all_coupling_map(coupling_map: CouplingMap) -> bool:
    # ! coupling_map.graph is a directed coupling map
    if coupling_map.size() * (coupling_map.size() - 1) == len(coupling_map.get_edges()):
        return True
    return False


def qiskit_O3_all2all(circ: qiskit.QuantumCircuit) -> qiskit.QuantumCircuit:
    from itertools import combinations
    for q0, q1 in combinations(range(circ.num_qubits), 2):
        circ.cx(q0, q1)
        circ.cx(q0, q1)
    circ = qiskit.transpile(circ, optimization_level=3, basis_gates=['u1', 'u2', 'u3', 'cx'])
    return circ

def optimize_with_mapping(circ: qiskit.QuantumCircuit, coupling_map: CouplingMap) -> qiskit.QuantumCircuit:
    """By default, we use Qiskit's O3 compiler to performa hardware-aware tranpilation and optimization"""
    # circ = qiskit_O3_all2all(circ)  # since input is logical circuit, we can first ally an all2all Qiskit O3
    circ = qiskit.transpile(circ, optimization_level=3,
                            basis_gates=['u1', 'u2', 'u3', 'cx'],
                            coupling_map=coupling_map, layout_method='sabre')
    return circ

def qiskit_pass(paulis: List[str], coeffs: List[float], coupling_map: CouplingMap = All2all, with_O3: bool = False) -> qiskit.QuantumCircuit:
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import PauliEvolutionGate
    from qiskit.quantum_info import SparsePauliOp
    from qiskit.transpiler.passes import HighLevelSynthesis, HLSConfig  

    n = len(paulis[0])  # number of qubits
    op = SparsePauliOp(paulis, coeffs)
    qc = QuantumCircuit(n)
    qc.append(PauliEvolutionGate(op), reversed(range(n)))  

    hls_config = HLSConfig(PauliEvolution=[  
        ("rustiq", {  
            "optimize_count": True,      # 优化双量子比特门数量  
            "preserve_order": False,     # 不保持 Pauli 项顺序  
            "upto_phase": True,         # 允许全局相位差异  
            "upto_clifford": False,     # 合成最终 Clifford 算子 (If True, 类似把尾端Clifford吸收进最终measurement) 
            "resynth_clifford_method": 1  # 使用 Qiskit 贪心合成 （If 2，类似把尾端Clifford吸收进最终measurement）
        })  
    ])
    hls_pass = HighLevelSynthesis(hls_config=hls_config)
    qc = hls_pass(qc)

    if is_all2all_coupling_map(coupling_map):
        if with_O3:
            qc = qiskit_O3_all2all(qc)
    else:
        qc = optimize_with_mapping(qc, coupling_map)

    return qc

In [11]:
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate


def rl_pass(paulis, coeffs, coupling_map: CouplingMap):
    n = len(paulis[0])  # number of qubits
    op = SparsePauliOp(paulis, coeffs)
    qc = QuantumCircuit(n)
    qc.append(PauliEvolutionGate(op), reversed(range(n)))
    qc = qc.decompose()
    
    ai_passmanager = PassManager([  
        # First, route the circuit to the target backend topology  
        AIRouting(coupling_map=coupling_map, optimization_level=3, layout_mode="optimize"),  
        
        # Collect Pauli Network blocks (H, S, SX, CX, RX, RY, RZ gates)  
        # Supports up to 6-qubit blocks  
        CollectPauliNetworks(  
            do_commutative_analysis=True,  
            min_block_size=4,  
            max_block_size=6,  
            num_reps=10  
        ),  
        
        # Apply RL-based synthesis to optimize the collected blocks  
        # Uses reinforcement learning models trained to minimize gate count and depth  
        AIPauliNetworkSynthesis(  
            coupling_map=coupling_map,  # Target backend for connectivity constraints  
            replace_only_if_better=True,  # Only replace if RL synthesis improves the circuit  
            max_threads=10,  # Parallel synthesis requests  
        )  
    ])
    qc_rl = ai_passmanager.run(qc)
    return qc_rl

In [14]:
with open('../benchmarks/uccsd_json/CH2_frz_BK_sto3g.json', 'r') as f:
    data = json.load(f)

paulis = []
coeffs = []
groups = group_paulis_and_coeffs(data['paulis'], data['coeffs'])
for indices, (paulis_part, coeffs_part) in groups.items():
    if len(indices) >= 6:
        continue
    paulis.extend(paulis_part)
    coeffs.extend(coeffs_part)

paulis = paulis[:100]
coeffs = coeffs[:100]

In [15]:
qc_rustiq = qiskit_pass(paulis, coeffs, coupling_map=gene_square_coupling_map(data['num_qubits']))
print_circ_info(qc_rustiq, "Qiskit + RustiQ")

                      Qiskit + RustiQ                       
┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓
┃ num_qubits ┃ num_gates ┃ num_2q_gates ┃ depth ┃ depth_2q ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩
│ 12         │ 1062      │ 671          │ 630   │ 477      │
└────────────┴───────────┴──────────────┴───────┴──────────┘

In [16]:
n = data['num_qubits']  # number of qubits
op = SparsePauliOp(paulis, coeffs)
qc = QuantumCircuit(n)
qc.append(PauliEvolutionGate(op), reversed(range(n)))
print_circ_info(qc.decompose(), "Original Circuit (logical-level, trivial synthesis)")

    Original Circuit (logical-level, trivial synthesis)     
┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓
┃ num_qubits ┃ num_gates ┃ num_2q_gates ┃ depth ┃ depth_2q ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩
│ 12         │ 1604      │ 758          │ 827   │ 600      │
└────────────┴───────────┴──────────────┴───────┴──────────┘

In [17]:
# qc_rl = rl_pass(paulis, coeffs, coupling_map=CouplingMap.from_full(data['num_qubits']))
qc_rl = rl_pass(paulis, coeffs, coupling_map=gene_square_coupling_map(data['num_qubits']))
print_circ_info(qc_rl, "Qiskit + RL-based Pauli Network Synthesis")

INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running P

         Qiskit + RL-based Pauli Network Synthesis          
┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓
┃ num_qubits ┃ num_gates ┃ num_2q_gates ┃ depth ┃ depth_2q ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩
│ 12         │ 1473      │ 762          │ 855   │ 527      │
└────────────┴───────────┴──────────────┴───────┴──────────┘

In [18]:
qc_rl.draw(fold=-1)

┌───┐     ┌───┐┌───┐                                                                 ┌────┐┌───┐      ┌───┐     ┌───┐┌────┐                                     ┌───┐                              ┌───┐                               ┌───┐                                       ┌───┐                                       ┌───┐                   ┌───┐    ┌───┐                                                     ┌───┐            ┌───┐             ┌───┐┌───┐                    ┌───┐                                                                                                                                                                                                                                                                                                                                                    ┌────┐┌──────┐┌───┐          ┌───┐                                            ┌────┐┌──────┐┌───┐     ┌───┐                                                            ┌───┐┌────┐┌──────┐┌───┐                                            ┌──────┐┌───┐    ┌───┐                                                                                                                                                                                                                            ┌──────┐┌───┐                                                                   ┌───┐                                                              ┌────┐              ┌───┐      ┌───┐ ┌───┐                          ┌────┐                   ┌───┐  ┌───┐                                                              ┌───┐                                ┌───┐                                                     ┌───┐                                     ┌───┐                                      ┌───┐         ┌────┐┌───┐                                    ┌───┐           ┌───┐    ┌───┐          ┌───┐┌───┐┌────┐          ┌───┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 ┌───┐                                                                                 ┌────┐                ┌───┐                                                                ┌───┐                                           ┌───┐                                               ┌───┐                    ┌───┐                    ┌───┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [19]:
from phoenix.models import HamiltonianModel

def phoenix_pass(paulis: List[str], coeffs: List[float],
                 order_blocks: bool = True,
                 efficient: bool = False,
                 coupling_map: CouplingMap = All2all,
                 with_O3: bool = False) -> qiskit.QuantumCircuit:
    """Phoenix's high-level optimization"""
    ham = HamiltonianModel(paulis, coeffs)
    circ = ham.phoenix_circuit(order_blocks=order_blocks, efficient=efficient)

    # from phoenix.utils import passes
    # circ = passes.remove_front_cliffords(circ)
    # circ = passes.remove_last_cliffords(circ)

    qc = circ.to_qiskit()
    if is_all2all_coupling_map(coupling_map):
        if with_O3:
            qc = qiskit_O3_all2all(qc)
    else:
        qc = optimize_with_mapping(qc, coupling_map)

    return qc

In [20]:
qc_phoenix = phoenix_pass(paulis, coeffs, coupling_map=gene_square_coupling_map(data['num_qubits']))
print_circ_info(qc_phoenix, "Phoenix")

                          Phoenix                           
┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓
┃ num_qubits ┃ num_gates ┃ num_2q_gates ┃ depth ┃ depth_2q ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩
│ 12         │ 1104      │ 566          │ 585   │ 370      │
└────────────┴───────────┴──────────────┴───────┴──────────┘

In [99]:
num_qubits = 6
from qiskit.circuit.library import EfficientSU2
qc_su2 = EfficientSU2(num_qubits, entanglement="linear", reps=2).decompose()


In [100]:
print_circ_info(qc_su2, "EfficientSU2 Circuit")

                    EfficientSU2 Circuit                    
┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓
┃ num_qubits ┃ num_gates ┃ num_2q_gates ┃ depth ┃ depth_2q ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩
│ 6          │ 46        │ 10           │ 13    │ 7        │
└────────────┴───────────┴──────────────┴───────┴──────────┘

In [105]:
print_circ_info(qiskit.transpile(qc_su2, optimization_level=3,
                 basis_gates=['u1', 'u2', 'u3', 'cx'],
                 coupling_map=gene_square_coupling_map(num_qubits), layout_method='sabre'))

┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓
┃ num_qubits ┃ num_gates ┃ num_2q_gates ┃ depth ┃ depth_2q ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩
│ 6          │ 46        │ 10           │ 13    │ 7        │
└────────────┴───────────┴──────────────┴───────┴──────────┘